# Exercise 44 — `pathlib.Path`

## Concept

`pathlib` is the modern way to handle paths. Replace string concatenation and `os.path` functions with `Path` objects.

```python
from pathlib import Path

p = Path("data") / "raw" / "users.csv"   # OS-correct separator: data\raw\users.csv on Windows, data/raw/users.csv elsewhere

p.exists()          # True/False
p.is_file()         # is it an existing file?
p.is_dir()          # is it an existing directory?

p.parent            # Path('data/raw')
p.name              # 'users.csv'
p.stem              # 'users'
p.suffix            # '.csv'
p.parts             # ('data', 'raw', 'users.csv')

p.read_text(encoding="utf-8")
p.write_text("...", encoding="utf-8")
p.read_bytes() / p.write_bytes(...)

Path("out").mkdir(parents=True, exist_ok=True)    # safe directory creation
p.unlink(missing_ok=True)                          # delete file, no error if missing

list(Path("data").glob("*.csv"))                   # files matching pattern (current dir only)
list(Path("data").rglob("*.csv"))                  # recursive
```

Why `Path` over strings:
- Cross-platform separators (Windows vs Unix) handled for free
- Methods are organized as members of `Path` — easier to discover
- Returns `Path` objects, which compose nicely

In DE: every pipeline manipulates paths. Use `Path` everywhere; only call `str(p)` when an old API forces it.

## Setup — build a tiny directory tree

Run this cell once. It creates a small folder structure for the tasks below.

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("sandbox")
if ROOT.exists():
    shutil.rmtree(ROOT)

(ROOT / "raw").mkdir(parents=True)
(ROOT / "processed").mkdir(parents=True)
(ROOT / "raw" / "users.csv").write_text("id,name\n1,alice\n", encoding="utf-8")
(ROOT / "raw" / "orders.csv").write_text("id,total\n1,10\n", encoding="utf-8")
(ROOT / "raw" / "notes.txt").write_text("some notes", encoding="utf-8")
(ROOT / "processed" / "users.parquet").write_bytes(b"\x00\x00fake")
print("sandbox ready at", ROOT.resolve())


## Task 1 — Inspect a path

Write a function that, given a `Path`, returns a dict with `"name"`, `"stem"`, `"suffix"`, `"parent"` (as a string), and `"exists"` (bool).

```python
def path_info(p: Path) -> dict:
    ...
```

Use the `Path` accessors — no string parsing.

Expected for `sandbox/raw/users.csv` (after the setup cell):
```
{
    "name": "users.csv",
    "stem": "users",
    "suffix": ".csv",
    "parent": str(Path("sandbox/raw")),   # OS-specific separator
    "exists": True,
}
```

In [ ]:
from pathlib import Path

def path_info(p: Path) -> dict:
    pass  # TODO

p = Path("sandbox/raw/users.csv")
info = path_info(p)
assert info["name"] == "users.csv"
assert info["stem"] == "users"
assert info["suffix"] == ".csv"
assert info["parent"] == str(Path("sandbox/raw"))
assert info["exists"] is True
print("ok")


## Task 2 — List files matching a pattern

Write a function that returns the sorted list of CSV file names (just the `.name`, not the full path) under a given directory — **non-recursive**.

```python
def list_csvs(directory: Path) -> list[str]:
    ...
```

Use `Path.glob("*.csv")`. Returns a list of strings, sorted alphabetically, for determinism.

Expected:
- `list_csvs(Path("sandbox/raw"))` → `["orders.csv", "users.csv"]`
- `list_csvs(Path("sandbox/processed"))` → `[]` (no CSVs there — only parquet)

In [ ]:
from pathlib import Path

def list_csvs(directory: Path) -> list[str]:
    pass  # TODO

assert list_csvs(Path("sandbox/raw")) == ["orders.csv", "users.csv"]
assert list_csvs(Path("sandbox/processed")) == []
print("ok")


## Task 3 — Ensure output directory and move data

Write a function that, given a source CSV path and a destination directory:
- Creates the destination directory if it doesn't exist (including intermediate dirs)
- Reads the source's text content
- Writes a `.copy.csv` file (same stem + `.copy.csv`) inside the destination
- Returns the new `Path`

```python
def stage_copy(src: Path, dst_dir: Path) -> Path:
    ...
```

Example: `stage_copy(Path("sandbox/raw/users.csv"), Path("sandbox/staging"))` should create `sandbox/staging/users.copy.csv` with the same content as the source, and return that path.

Use `Path.mkdir(parents=True, exist_ok=True)` so re-running is safe.

In [ ]:
from pathlib import Path

def stage_copy(src: Path, dst_dir: Path) -> Path:
    pass  # TODO

new_path = stage_copy(Path("sandbox/raw/users.csv"), Path("sandbox/staging"))
assert new_path == Path("sandbox/staging/users.copy.csv")
assert new_path.exists()
assert new_path.read_text(encoding="utf-8") == "id,name\n1,alice\n"

# safe to re-run
new_path2 = stage_copy(Path("sandbox/raw/users.csv"), Path("sandbox/staging"))
assert new_path2 == new_path
print("ok")
